# Clinical Trial Radar — Notebook pentru documentație și continuarea proiectului
Această foaie demonstrează: setup, structură, cod, teste, generare doc și CI pentru prototipul Stage 1.

## 1. Configurare mediu și dependențe

Instalare și verificare:

```bash
python -m venv .venv
.venv\Scripts\activate
pip install -r requirements.txt
python -c "import sys, pandas; print(sys.version); print('pandas', pandas.__version__)"
```

## 2. Structura proiectului și fișiere cheie

Schema de directoare:

- main.py
- ct_radar/
  - __init__.py
  - ingest.py
  - normalize.py
  - metrics.py
  - visualize.py
- data/
- docs/

Celulă pentru a valida prezența fișierelor (rulare simplă):

In [ ]:
# Validate structure
import os
base = os.getcwd()
paths = ["main.py","ct_radar/__init__.py","ct_radar/ingest.py","docs/README.md"]
for p in paths:
    print(p, 'OK' if os.path.exists(p) else 'MISSING')

## 3. Generare documentație automată (Sphinx)

Instalare și inițializare (exemplu Sphinx):

```bash
pip install sphinx sphinx-rtd-theme
sphinx-quickstart docs_src
# edit docs_src/conf.py to add project metadata and extensions
sphinx-build -b html docs_src docs/_build/html
```

## 4. Încărcare și validare configurații

Exemplu: citire YAML și validare cu pydantic


In [ ]:
# example_config.yaml
example = '''
registry_base: https://clinicaltrials.gov
default_phase: Phase 2
max_results: 200
'''

# validate with pydantic (if available)
try:
    from pydantic import BaseModel
    import yaml

    class ConfigModel(BaseModel):
        registry_base: str
        default_phase: str
        max_results: int

    cfg = yaml.safe_load(example)
    c = ConfigModel(**cfg)
    print('Config valid:', c)
except Exception as e:
    print('Validation skipped or failed:', e)


## 5. Implementare module principale (schelet și funcționalitate)

Exemplu: import și apeluri la module existente (`ct_radar`).

In [ ]:
# Example: run a small ingest (uses network)
try:
    from ct_radar import fetch_and_save_studies, compute_metrics_from_path
    print('Modules imported')
    # path = fetch_and_save_studies('type 2 diabetes', 'Phase 2', max_results=20, out_dir='data/raw')
    # print('Ingest saved to', path)
except Exception as e:
    print('Import or network call skipped:', e)


## 6. Exemplu de rulare a componentei principale

În notebook: încărcare metrics și afișare trend + hartă (dacă datele există).

In [ ]:
try:
    import pandas as pd
    from ct_radar import compute_metrics_from_path, plot_trend, plot_country_choropleth
    metrics = compute_metrics_from_path('data/raw')
    print('n_total_studies:', metrics.get('n_total_studies'))
    if metrics.get('trend_by_start_year'):
        df_trend = pd.DataFrame(metrics['trend_by_start_year'])
        display(df_trend)
    if metrics.get('crowdedness'):
        df_crowd = pd.DataFrame(metrics['crowdedness'])
        display(df_crowd.head())
except Exception as e:
    print('Skipping runtime visualization in notebook:', e)


## 7. Testare unități și integrare (pytest)

Exemple:

```bash
pip install pytest
pytest tests/
```

Exemplu de test (tests/test_normalize.py):

```python
from ct_radar.normalize import normalize_disease

def test_normalize():
    assert normalize_disease('t2dm') == 'diabetes mellitus'
```


## 8. Integrare cu VS Code: task-uri și debug

Sugestii ` .vscode/launch.json` și `tasks.json` pot rula `main.py` și `pytest` din editor.


## 9. Debugging, logging și profilare

- Configurabil: `logging` cu nivele
- Profilare: `python -m cProfile -o profile.out main.py` și `snakeviz profile.out`

## 10. CI/CD: GitHub Actions minimal

Exemplu minimal `/.github/workflows/ci.yml`:

```yaml
name: CI
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: '3.9'
      - name: Install deps
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
      - name: Run tests
        run: pytest -q
      - name: Build docs
        run: |
          pip install sphinx
          sphinx-build -b html docs_src docs/_build/html
```

## 11. Versiuni, packaging și release

Exemplu folosind `setuptools` / `pyproject.toml`:

```bash
python -m build
pip install dist/*.whl
```